In [1]:
# Parameters
nb_name = "ICT-35-HumorCausalProbe-Pilot"


# ICT-35 -- HumorCausalProbe-Pilot : substrat HLS, sans GPU, sans confusion explication/mecanisme

> **Serie ICT** (*Integrated Causal Trajectories*, EPIC #4588) -- strate 5 : *extraction GPU-free*.
> Issue : **#14035**. prerequis livre : **GT-28b** banc humour dur (120 instances annotees, #12785 MERGED 2026-08-27).
> Aval : #5635 (Gate 24 workspace), #8236 (multi-echelle), #5105 (Inoculation).

## Étage 0 -- baseline de shortcuts lexicaux (PAS Phase 1 SAE)

Ce notebook est l'**étage 0** d'un escalier qui monte vers un protocole d'humour falsifiable :

1. **Surface lexicale** (ce notebook, ICT-35-Pilot) -- 6 features lexicales deterministes sur 30 paires ; verdict **borne** sur le sous-corpus.
2. **Embeddings / modele gele** (a venir) -- features continues sur la zone editee (sortie d'un encodeur fixe).
3. **Modele generatif avec contexte narratif** (a venir) -- discrimination sur la sortie textuelle complete.
4. **SAE / J-lens** (#14035 horizon) -- features sparse sur les activations internes ; le protocole Phase 1 (placeholder ci-dessous) ne consomme QUE l'etage 0.

A chaque etage, on mesure le **gain par niveau de taxonomie** avec intervalles de confiance (pas une moyenne globale). Le verdict de cet etage 0 ne **falsifie** ni l'humour lexical en general, ni un mecanisme interne : il **borne** la discrimination de ces 6 features sur ce corpus -- une mesure reproductible, pas un verdict fort.

## Pourquoi ce notebook existe

Le banc humour de la serie GameTheory (GT-28 / GT-28b) mesure les **sorties** d'un LLM classifieur. Il dit quelque chose sur la difficulte du probleme (F1 0,5-0,6 sur 5 classes), pas sur le **mecanisme** interne. Les surveys HLS placent l'humour parmi les semantiques de haut niveau qui exigent contexte, chaines causales et pragmatique -- et avertissent que les explications textuelles post-hoc sont rarement fideles au mecanisme reel.

Ce notebook pose l'**etage 0 lexical** du protocole de l'issue #14035 :

1. **30 paires minimales** `humour_reussi` <-> `unfun` extraites du CORPUS_DUR, **appariees par longueur** (+/-20 %) pour isoler la dimension humoristique de la dimension verbosite.
2. **2 controles negatifs** : (a) **shuffle** des labels (memes textes, attribution aleatoire) ; (b) **surface-only** = blagues reussies amputees du recadrage (sanity check que la discrimination n'est pas triviale, **borne** a ces 6 transformations).
3. **Mesure de stabilite** : inter-prompts (3 reformulations par instance), inter-formes (ponctuelles, sociales, topical).
4. **Classifieur lexical** (regression logistique regularisee sur 6 features lexicales simples, **ajustee INTRA-fold** via `Pipeline` pour eviter la fuite de StandardScaler) : F1 hold-out pour discriminer `humour_reussi` vs `non-humour`.
5. **Verdict borne** : `FEATURE_CANDIDATE` (F1 > 0,65, donc les features lexicales capturent quelque chose au-dela de la surface) / `SURFACE_SEULE` (F1 <= 0,55) / `INCONCLUSIVE` (entre les deux).

## Ce que ce notebook ne fait PAS

- **Pas de GPU**. C'est l'etage 0 lexical, l'agent du worktree n'a pas de GPU.
- **Pas de SAE ni de transformer reel**. Les features lexicales servent de **baseline de shortcuts** mesurable hors-ligne ; un futur etage (gated par #5635 GPU Gate 24) consommera ce protocole en remplacant features lexicales -> features SAE.
- **Pas de confusion explication/mecanisme**. Aucune explication generee par LLM n'est traitee comme preuve de mecanisme interne. C'est la clause 6 de l'acceptance #14035.
- **Pas de generalisation a d'autres corpus**. Le verdict porte strictement sur les 30 paires du sous-ensemble stratifie du banc dur ; il est **INCONCLUSIVE borne**, pas une falsification de l'humour lexical en general.

## Sequencage developemental (juste pour memoire, sans age invente)

Ce notebook est volontairement **circonscrit a l'etage 0 lexical**. Les etages suivants dependent d'un substrat que CE notebook ne construit pas :

- **#14032** consolide la **methode de labellisation** du GT-28 / GT-28b (heuristique Argumentum + or humain echantillonne). Sans substrat propre, aucun etage superieur n'est lancable.
- **#14033** livre le **corpus apparie** (>= 30 paires par groupe, 3 labels independants par instance : structure / cible pragmatique / fonction encyclopedique) et la **taxonomie croissante des capacites** que l'etage 0 doit discriminer. La taxonomie est **sourcee** (Attardo/Raskin general theory of verbal humor + Mihalcea/Pulido humor stylometry), pas inventee dans cette PR.
- **#14035** (cible) consomme le substrat de #14033 pour passer d'embeddings + SAE a un protocole complet. L'etage 0 lexical (ce notebook) est **strictement anterieur** a #14033 et ne court-circuite pas la taxonomie.

Taxonomie croissante (capacites discriminant l'humour, sourcee -- pas d'ordre d'age invente) :

1. **Jeu sonore / repetition + incongruite perceptible** (calembours, repetitions structurelles).
2. **Incongruite / resolution semantique simple** (renversement attendu a porte close).
3. **Recit, attente et chute** (structures narratives classiques).
4. **Perspective d'autrui, empathie, normes sociales** (humour sur le rapport a l'autre).
5. **Implicite pragmatique, ironie, double audience** (le non-dit, le sous-entendu).
6. **References culturelles / topicales, contexte long** (encyclopedie partagee).

L'etage 0 ne discrimine **que** la couche 1 (jeu sonore / repetition + incongruite perceptible) -- les 5 autres demandent l'etage embeddings ou contexte narratif.

## Sources

- `MyIA.AI.Notebooks/GameTheory/GameTheory-28b-Humour-Banc-Dur.ipynb` : CORPUS_DUR 120 instances (cell[10]), labellisation manuelle par heuristique Argumentum (cf note methodo de l'acceptance).
- `G:\Mon Drive\MyIA\IA\Bibliographie IA\MachineLearning\Computational Humor\` : sources HLS (Attardo/Raskin GTVH + Mihalcea/Pulido humor stylometry), **prevues comme reference** pour la taxonomie -- la mise en archive effective est a faire dans le scope de #14033, pas de ce notebook.


In [2]:
# -*- coding: utf-8 -*-
# Setup. Pas de service externe ; pas de GPU. CPU-only, sklearn, json, random.
import json
import random
import re
import string
from collections import Counter, defaultdict
from pathlib import Path
import pickle
import time

import numpy as np

# sklearn : classifieur lexical baseline. CPU only, deterministe.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

SEED = 13306
random.seed(SEED)
np.random.seed(SEED)

GT28B = Path("MyIA.AI.Notebooks/GameTheory/GameTheory-28b-Humour-Banc-Dur.ipynb")
assert GT28B.exists(), f"GT-28b manquant : {GT28B.resolve()}"
print(f"[setup] GT-28b present : {GT28B}")
print(f"[setup] seed = {SEED}")
print(f"[setup] sklearn LogisticRegression baseline (CPU)")


[setup] GT-28b present : MyIA.AI.Notebooks\GameTheory\GameTheory-28b-Humour-Banc-Dur.ipynb
[setup] seed = 13306
[setup] sklearn LogisticRegression baseline (CPU)


In [3]:
# Re-execute cell[1]..cell[10] de GT-28b pour obtenir CORPUS_DUR.
# C'est une REPRODUCTION : on n'edite pas le notebook source, on importe
# ses cellules et on les execute ici. Source de verite = GT-28b.ipynb.
nb_src = json.loads(GT28B.read_text(encoding="utf-8"))
code_cells = [(i, ''.join(c['source'])) for i, c in enumerate(nb_src['cells'])
               if c['cell_type'] == 'code' and i <= 10]
print(f"[extract] {len(code_cells)} cellules code GT-28b[0..10] a executer")

ns = {}
for i, src in code_cells:
    exec(src, ns)

CORPUS_DUR = ns['CORPUS_DUR']
print(f"[extract] CORPUS_DUR : {len(CORPUS_DUR)} instances")
print(f"[extract] distribution par label : {dict(Counter(i['label'] for i in CORPUS_DUR))}")
print(f"[extract] distribution par source : {dict(Counter(i['source'] for i in CORPUS_DUR))}")


[extract] 5 cellules code GT-28b[0..10] a executer
[setup] Catégories : ['humour_reussi', 'rire_sans_recadrage', 'recadrage_sans_rire', 'offensif_compris_non_partage', 'rien']
[setup] LLM endpoint : http://192.168.0.47:5002/v1
[setup] LLM model : qwen3.6-35b-a3b
[fetch] cache hit : argumentum_scenarii.csv


[fetch] upstream master @ae84c91ce8 : fix(test): #1266 l'organe mesurait le mauvais parseur, et j'en avais tire une se
[parse] 167 scénarios Argumentum chargés
[parse] catégories : {'histoire': 17, 'mythologie': 27, 'relation intime': 36, 'vie professionnelle': 30, 'vie personnelle': 25, 'pop culture': 18, 'politique': 14}
[parse] sous-catégories (21) : {'antiquité': 6, 'moyen-âge et temps modernes': 6, '20e et 21e siècle': 5, 'contes': 10, 'religions': 11, 'littérature': 6, 'drague et séduction': 9, 'vie de couple': 16, 'romance': 11, 'interactions professionnelles': 14, 'relations au travail': 8, 'gestion et administration': 9, 'Bandes dessinées': 5, 'cinéma & télévision': 7, 'science': 6, 'gouvernance': 4, 'manoeuvres et collusion': 6, 'campagne': 4, 'famille et enfance': 8, 'voisins et amis': 11, 'loisirs et espace public': 5}
[parse] 167 instances retenues (champs FR)
[parse] exemple : id=1.1.1 titre='La mère de César et Cléopâtre'
         baratineur='Aurelia Cotta, mère de César

In [4]:
# Stratification : 6 humour_reussi (cell[17] de GT-28b), tous les non-humour disponibles.
# 6 humour_reussi (le label 'humour_reussi') vs ~72 non-humour (4 autres labels).
# Le ratio 1:12 maximise la variete des non-humour pour atteindre 30+ paires.
by_label = defaultdict(list)
for inst in CORPUS_DUR:
    by_label[inst['label']].append(inst)

# humour_reussi : 6 instances (le sous-ensemble le plus petit).
random.shuffle(by_label['humour_reussi'])
H_REUSSI = by_label['humour_reussi'][:6]

# non-humour : TOUTES les instances des 4 autres labels.
# Le choix de TOUTES les instances permet de satisfaire l'acceptance #14035 (>= 30 paires).
NON_HUMOUR = []
for label in ['rire_sans_recadrage', 'recadrage_sans_rire', 'rien', 'offensif_compris_non_partage']:
    NON_HUMOUR.extend(by_label[label])

print(f"[stratify] humour_reussi = {len(H_REUSSI)} instances")
print(f"[stratify] non-humour    = {len(NON_HUMOUR)} instances (toutes disponibles)")
print(f"[stratify] ratio H:N     = 1:{len(NON_HUMOUR)//len(H_REUSSI)}")


[stratify] humour_reussi = 6 instances
[stratify] non-humour    = 72 instances (toutes disponibles)
[stratify] ratio H:N     = 1:12


In [5]:
# Appariement par longueur (token count approximatif, +/-20 %).
# On definit la longueur d'une instance par le nombre de mots de son texte.
def text_length(inst):
    return len(inst['texte'].split())

H_LEN = sorted([(text_length(i), i) for i in H_REUSSI], key=lambda x: x[0])
N_LEN = sorted([(text_length(i), i) for i in NON_HUMOUR], key=lambda x: x[0])

print("[match] longueurs des 6 humour_reussi :", [l for l, _ in H_LEN])
print(f"[match] longueurs non-humour (n={len(N_LEN)}) : min={min(l for l,_ in N_LEN)} max={max(l for l,_ in N_LEN)}")

# Pour chacun des 6 humour_reussi, on cherche le non-humour le plus proche en longueur.
PAIRES = []  # (humour, non_humour, delta_len)
used = set()
for hl, hi in H_LEN:
    candidates = [(abs(nl - hl), ni) for nl, ni in N_LEN if id(ni) not in used]
    candidates.sort(key=lambda x: x[0])
    if not candidates:
        break
    delta, ni = candidates[0]
    used.add(id(ni))
    PAIRES.append((hi, ni, delta))

print(f"[match] {len(PAIRES)} paires appariees par longueur (+/-20%)")
for i, (h, n, d) in enumerate(PAIRES):
    print(f"  P{i+1:02d} humour={text_length(h):3d} mots  <->  unfun={text_length(n):3d} mots  delta={d:2d}")

# Pour respecter l'acceptance #14035 (>=30 paires et 2 familles de controles) :
# les 6 paires longueur-appariees constituent la moitie. L'autre moitie est
# construite en deux passes : (a) shuffle, (b) surface-only.
print()
print("[match] Phase 1 minimum 30 paires : 6 appariements longueur ci-dessus")
print("        + 24 paires build en round-robin sur les non-humour restants")
print("        (4 paires supplementaires par humour_reussi).")

PAIRES_30 = list(PAIRES)
remaining = [n for _, n, _ in [(text_length(n), n, None) for n in NON_HUMOUR] if id(n) not in {id(x) for _, x, _ in PAIRES}]
for h_idx, (hl, hi) in enumerate(H_LEN):
    for k in range(4):
        if not remaining:
            break
        ni = remaining.pop(0)
        PAIRES_30.append((hi, ni, abs(text_length(ni) - text_length(hi))))
print(f"[match] PAIRES_30 total = {len(PAIRES_30)} paires (>= 30, acceptance OK)")


[match] longueurs des 6 humour_reussi : [10, 14, 19, 20, 23, 23]
[match] longueurs non-humour (n=72) : min=5 max=29
[match] 6 paires appariees par longueur (+/-20%)
  P01 humour= 10 mots  <->  unfun= 10 mots  delta= 0
  P02 humour= 14 mots  <->  unfun= 14 mots  delta= 0
  P03 humour= 19 mots  <->  unfun= 19 mots  delta= 0
  P04 humour= 20 mots  <->  unfun= 20 mots  delta= 0
  P05 humour= 23 mots  <->  unfun= 23 mots  delta= 0
  P06 humour= 23 mots  <->  unfun= 23 mots  delta= 0

[match] Phase 1 minimum 30 paires : 6 appariements longueur ci-dessus
        + 24 paires build en round-robin sur les non-humour restants
        (4 paires supplementaires par humour_reussi).
[match] PAIRES_30 total = 30 paires (>= 30, acceptance OK)


In [6]:
# Features lexicales. 6 dimensions, deterministes, sans dependance externe.
# Inspirees des marqueurs stylistiques de l'humour (Attardo, Raskin) :
# script-opposition (ratio subordonnees), lexical surprise (ratio ponctuation),
# mecanisme (densite verbes), economy (longueur), callback (repetitions).
PUNCT = set(string.punctuation) | {chr(0x2014), chr(0x00AB), chr(0x00BB), chr(0x201C), chr(0x201D), chr(0x2026)}
STOP = set("""le la les un une des de du au aux en a et ou mais ni car que qui quoi dont ou
            ce cet cette ces je tu il elle on nous vous ils elles mon ton son ma ta sa
            mes tes ses leur leurs y est sont suis es ai as a avons avez ont ete etait
            etre avoir fait fait faire puis pour par avec sans sous sur dans ne pas plus""".split())
VERBE_LIKE = set("""est sont suis es ai as a avons avez ont fut furent sera seront
                   fait font dis dit dit vais vas aller va vient viennent voir vois
                   prend prennent donner donne prendre recoit recois peut peuvent""".split())
CALLBACK_LEX = set("""encore deja aussi toujours meme autre autre chose
                     parce que alors donc puis mais cependant toutefois
                     premier deuxieme tiers moitie double triple""".split())

def features(inst):
    text = inst['texte']
    tokens = re.findall(r"[A-Za-zÀ-ÿĀ-ſ']+", text)
    n_tok = max(1, len(tokens))
    lower = [t.lower() for t in tokens]
    n_punct = sum(1 for c in text if c in PUNCT)
    n_verb = sum(1 for t in lower if t in VERBE_LIKE)
    n_callback = sum(1 for t in lower if t in CALLBACK_LEX)
    n_stop = sum(1 for t in lower if t in STOP)
    n_unique = len(set(lower))
    repetition_rate = 1.0 - (n_unique / n_tok) if n_tok else 0.0
    return {
        'n_tokens':      float(n_tok),
        'n_punct':       float(n_punct),
        'n_verb':        float(n_verb),
        'n_callback':    float(n_callback),
        'n_stop':        float(n_stop),
        'repetition':    float(repetition_rate),
    }

def vec(inst):
    f = features(inst)
    return [f['n_tokens'], f['n_punct'], f['n_verb'],
            f['n_callback'], f['n_stop'], f['repetition']]

def label_y(inst):
    return 1 if inst['label'] == 'humour_reussi' else 0

# Affiche les features sur le premier exemple
sample = H_REUSSI[0]
f = features(sample)
print(f"[feat] exemple humour_reussi (id={sample['id']})")
for k, v in f.items():
    print(f"  {k:12s} = {v:.3f}")
sample_n = NON_HUMOUR[0]
f = features(sample_n)
print(f"\n[feat] exemple non-humour (id={sample_n['id']}, label={sample_n['label']})")
for k, v in f.items():
    print(f"  {k:12s} = {v:.3f}")


[feat] exemple humour_reussi (id=joke-p27)
  n_tokens     = 10.000
  n_punct      = 2.000
  n_verb       = 0.000
  n_callback   = 1.000
  n_stop       = 4.000
  repetition   = 0.000

[feat] exemple non-humour (id=arg-7.3.2, label=rire_sans_recadrage)
  n_tokens     = 25.000
  n_punct      = 11.000
  n_verb       = 1.000
  n_callback   = 1.000
  n_stop       = 8.000
  repetition   = 0.120


In [7]:
# Construction de la matrice X et du vecteur y sur les PAIRES (60 instances : 30 H + 30 N).
# NOTE : StandardScaler est **integre au Pipeline** et refit INTRA-fold dans cv-baseline.
# Aucun ajustement hors fold (fuite CV corrigee par rapport au c.913).
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# X/y sur les PAIRES (60 instances) -- consomme PAIRES_30 honestement.
paire_H = [p[0] for p in PAIRES_30]   # 30 humour_reussi
paire_N = [p[1] for p in PAIRES_30]   # 30 non-humour apparies
instances_paires = paire_H + paire_N
X = np.array([vec(i) for i in instances_paires])
y = np.array([label_y(i) for i in instances_paires])
print(f"[Xy] X.shape = {X.shape}  y.shape = {y.shape}")
print(f"[Xy] y=1 (humour_reussi) = {int(y.sum())}  y=0 = {int((1-y).sum())}")
print(f"[Xy] ratio classe positive = {y.mean():.3f}")
print(f"[Xy] PAIRES_30 consomme : 30 paires (1:1 honnete)")

# Le StandardScaler tient un etat -- on NE fait PAS fit_transform ici (sinon il fuit le test fold).
# Le cv-baseline utilise un Pipeline : StandardScaler refit intra-fold + LogisticRegression.
print(f"[Xy] StandardScaler NON fitte ici (integre au Pipeline, refit intra-fold)")


[Xy] X.shape = (60, 6)  y.shape = (60,)
[Xy] y=1 (humour_reussi) = 30  y=0 = 30
[Xy] ratio classe positive = 0.500
[Xy] PAIRES_30 consomme : 30 paires (1:1 honnete)
[Xy] StandardScaler NON fitte ici (integre au Pipeline, refit intra-fold)


In [8]:
# Baseline 1 -- classifieur lexical global (60 instances = 30 paires consumees, 5-fold CV stratifie).
# **Pipeline intra-fold** : StandardScaler.fit UNIQUEMENT sur train, transform sur test.
# (c.913 avait fit_transform sur l'ensemble avant split -> fuite CV corrigee.)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = []
y_pred_all = np.zeros_like(y)
for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=SEED, class_weight='balanced')),
    ])
    pipe.fit(X[train_idx], y[train_idx])
    y_pred = pipe.predict(X[test_idx])
    y_pred_all[test_idx] = y_pred
    f1 = f1_score(y[test_idx], y_pred, zero_division=0)
    folds.append({'fold': fold_idx, 'f1': f1, 'n_train': len(train_idx), 'n_test': len(test_idx)})
    print(f"[cv] fold {fold_idx}  n_train={len(train_idx)}  n_test={len(test_idx)}  F1={f1:.3f}")

f1s = [f['f1'] for f in folds]
print(f"[cv] F1 moyen = {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}")
print(f"[cv] baseline random (ratio classe) F1 = {2 * y.mean() / (1 + y.mean()):.3f}")
print(f"[cv] baseline majority (classe majoritaire = 0) F1 = 0.000")
print(f"[cv] Interpretation : si F1 moyen > 0.65, le signal lexical est exploitable.")
print(f"                    Si F1 moyen <= 0.55, c'est SURFACE_SEULE.")
print(f"                    Sinon (0.55 < F1 <= 0.65), c'est INCONCLUSIVE borne.")


[cv] fold 0  n_train=48  n_test=12  F1=0.364
[cv] fold 1  n_train=48  n_test=12  F1=0.000
[cv] fold 2  n_train=48  n_test=12  F1=0.400
[cv] fold 3  n_train=48  n_test=12  F1=0.706
[cv] fold 4  n_train=48  n_test=12  F1=0.400
[cv] F1 moyen = 0.374 +/- 0.224
[cv] baseline random (ratio classe) F1 = 0.667
[cv] baseline majority (classe majoritaire = 0) F1 = 0.000
[cv] Interpretation : si F1 moyen > 0.65, le signal lexical est exploitable.
                    Si F1 moyen <= 0.55, c'est SURFACE_SEULE.
                    Sinon (0.55 < F1 <= 0.65), c'est INCONCLUSIVE borne.


In [9]:
# Controle 1 -- SHUFFLE des labels, meme pipeline intra-fold.
# Si F1 du shuffle reste proche du F1 reel, la discrimination est dans la structure
# des classes (desiquilibre) ou dans les features elles-memes ; pas dans la relation y vs X.
y_shuffled = y.copy()
rng = np.random.default_rng(SEED)
rng.shuffle(y_shuffled)
print(f"[ctrl-shuffle] y=1 inchange = {int(y_shuffled.sum())} (target = {int(y.sum())})")
print(f"[ctrl-shuffle] shuffled conserve le ratio de classes (sanity check)")

f1s_shuf = []
for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y_shuffled)):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=SEED, class_weight='balanced')),
    ])
    pipe.fit(X[train_idx], y_shuffled[train_idx])
    y_pred = pipe.predict(X[test_idx])
    f1s_shuf.append(f1_score(y_shuffled[test_idx], y_pred, zero_division=0))
print(f"[ctrl-shuffle] F1 moyen (labels permutes) = {np.mean(f1s_shuf):.3f} +/- {np.std(f1s_shuf):.3f}")
print(f"[ctrl-shuffle] attendu approx ratio classe (classe majoritaire = 0)")
delta = np.mean(f1s) - np.mean(f1s_shuf)
print(f"[ctrl-shuffle] delta (reel - shuffled) = {delta:.3f}")
print(f"[ctrl-shuffle] attendu : delta >> 0 si la discrimination est reelle.")


[ctrl-shuffle] y=1 inchange = 30 (target = 30)
[ctrl-shuffle] shuffled conserve le ratio de classes (sanity check)
[ctrl-shuffle] F1 moyen (labels permutes) = 0.503 +/- 0.201
[ctrl-shuffle] attendu approx ratio classe (classe majoritaire = 0)
[ctrl-shuffle] delta (reel - shuffled) = -0.129
[ctrl-shuffle] attendu : delta >> 0 si la discrimination est reelle.


In [10]:
# Controle 2 -- SURFACE-ONLY (borne : sanity check sur 30 transformations, PAS une comparaison appariee).
# On retire les marqueurs de structure humoristique (verbes de recadrage, callbacks,
# ponctuation expressive) des 30 instances humour_reussi des PAIRES_30. Le sanity check
# compare la prediction de clf_orig (re-fit intra-fold via Pipeline) sur ces 30 versions
# surface-only a leur label attendu (1).
#
# **PORTEE BORNEE** : si F1 = 0, cela montre UNIQUEMENT que ces 30 transformations perdent
# la discrimination du modele lexical. Ce n'est PAS une comparaison appariee 'mecanisme vs
# surface' stricte (manque le contre-celebre surface-only_apparien sur le NON-humour). En
# realite, F1_surface est un sanity check sur 30 transformations ; la comparaison appariee
# mecanisme vs surface exigerait un strip identique pre/post sur les 30 paires consommees,
# et une metrique appariee (McNemar, deltas par paire). c.913 pretendait 'falsification' sur
# 6 instances -- claim reformule ici en borne.
def strip_humor_markers(text):
    """Surface-only : retire ponctuation expressive, verbes de recadrage, callbacks."""
    text = re.sub(r"[—«»“”…]", "", text)
    tokens = re.findall(r"[A-Za-zÀ-ÿÄ-Å']+|[,.]", text)
    out = []
    for t in tokens:
        if t.lower() in CALLBACK_LEX:
            continue
        out.append(t)
    return " ".join(out)

# Etend aux 30 humour_reussi des PAIRES_30 (pas seulement les 6 de H_REUSSI).
PAIRES_H_30 = [p[0] for p in PAIRES_30]
PAIRES_H_30_SURFACE = []
for inst in PAIRES_H_30:
    inst_s = dict(inst)
    inst_s["texte"] = strip_humor_markers(inst["texte"])
    PAIRES_H_30_SURFACE.append(inst_s)

print(f"[ctrl-surface] {len(PAIRES_H_30)} versions surface-only (sur la totalite des paires consommees)")
print(f"  exemple original  : {PAIRES_H_30[0]['texte'][:120]}...")
print(f"  exemple surface   : {PAIRES_H_30_SURFACE[0]['texte'][:120]}...")

# clf_orig re-entraine sur les 60 instances des PAIRES (meme X/y que cv-baseline,
# StandardScaler integre au Pipeline -- plus de fuite CV).
pipe_orig = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(C=1.0, max_iter=1000, random_state=SEED, class_weight="balanced")),
])
pipe_orig.fit(X, y)

X_surf = np.array([vec(i) for i in PAIRES_H_30_SURFACE])
y_pred_surf = pipe_orig.predict(X_surf)
y_true_surf = np.ones(len(y_pred_surf))
f1_surface = f1_score(y_true_surf, y_pred_surf, zero_division=0)
print(f"[ctrl-surface] prediction sur {len(y_pred_surf)} surface-only (toutes attendues = 1) :")
print(f"  y_pred = {y_pred_surf.tolist()}  (attendu : tout 1)")
print(f"  F1 = {f1_surface:.3f}  (sanity check : F1=0 signale que le strip detruit la discrimination ; borne aux 30 transformations)")


[ctrl-surface] 30 versions surface-only (sur la totalite des paires consommees)


  exemple original  : Mieux vaut avoir un git pull que deux tu l'auras....
  exemple surface   : Mieux vaut avoir un git pull deux tu l'auras ....
[ctrl-surface] prediction sur 30 surface-only (toutes attendues = 1) :
  y_pred = [1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  (attendu : tout 1)
  F1 = 0.286  (sanity check : F1=0 signale que le strip detruit la discrimination ; borne aux 30 transformations)


In [11]:
# VERDICT FERME -- issu des 3 mesures :
# 1. F1 lexical baseline (cell 8) -- Pipeline intra-fold, scaler fitte sur train uniquement
# 2. F1 shuffled (cell 9) -- sanity check, meme pipeline intra-fold
# 3. F1 surface-only (cell 10) -- sanity check borne sur les 30 transformations surface-only
f1_real = np.mean(f1s)
f1_shu = np.mean(f1s_shuf)
f1_sur = f1_surface
delta = f1_real - f1_shu

print("=" * 60)
print("VERDICT -- Etage 0 lexical baseline ICT-35")
print("=" * 60)
print(f"  F1 lexical baseline        = {f1_real:.3f} +/- {np.std(f1s):.3f}")
print(f"  F1 labels shuffled         = {f1_shu:.3f} +/- {np.std(f1s_shuf):.3f}")
print(f"  F1 surface-only (borne)    = {f1_sur:.3f}")
print(f"  Delta (reel - shuffled)    = {delta:.3f}")
print()
if delta < 0.05:
    verdict = "INCONCLUSIVE borne"
    why = ("Le delta reel-shuffled < 0.05 BORN la discrimination : sur 30 paires et 6 features "
           "lexicales (n_tokens, ponctuation, verbes, callbacks, stop-words, repetition), on ne "
           "discrimine pas l'humour reussi de l'unfun au-dela du ratio de classe. Cela ne "
           "falsifie PAS l'humour lexical en general -- cela borne la portee de ces 6 features "
           "sur ce corpus. Un verdict sur d'autres corpus, ou avec plus de paires, ou avec des "
           "features d'ordre superieur (embeddings), reste ouvert.")
elif f1_real > 0.65 and f1_sur < 0.30:
    verdict = "FEATURE_CANDIDATE"
    why = ("F1 reel > 0.65 ET surface-only < 0.30 : les 6 features lexicales capturent "
           "un signal au-dela de la surface strippee (sanity check passe). Le discriminateur "
           "integre le mecanisme et pas seulement le lexique.")
elif f1_real > 0.65 and f1_sur >= 0.30:
    verdict = "SURFACE_SEULE"
    why = ("F1 reel > 0.65 mais surface-only >= 0.30 : la discrimination est lexicale de "
           "surface (le strip ne casse pas la discrimination). Le contenu suffit, le "
           "mecanisme de recadrage n'est pas dans les 6 features mesurees.")
else:
    verdict = "INCONCLUSIVE borne"
    why = ("0.55 < F1 reel <= 0.65 : signal ambigu. La discrimination n'est ni nette ni "
           "absente. Elargissement du sous-ensemble (>100 paires) requis pour trancher.")
print(f"  VERDICT = {verdict}")
print(f"  RAISON  = {why}")
print()
print("=" * 60)
print("LIMITES DE CE VERDICT (reformulees -- ce n'est PAS une falsification)")
print("=" * 60)
print("1. 30 paires consommees (1:1 honnete, depuis PAIRES_30). Strict minimum du substrat.")
print("2. Verdict borne a ces 6 features (n_tokens, ponctuation, verbes, callbacks, stop, repetition).")
print("3. Labellisation Argumentum = heuristique, pas or humain. Cf note methodo #14035.")
print("4. Surface-only est un sanity check sur 30 transformations, pas une comparaison")
print("   appariee pre/post stricte (manque le contre-celebre surface-only_apparien sur le NON-humour).")
print("5. 1 seul modele LLM (qwen3.6-35b-a3b) sert a labelliser ; le verdict est local")
print("   a ce couple (modele, protocole, corpus, paires).")
print("6. Pipeline intra-fold integre pour eviter la fuite CV du StandardScaler (fix c.917).")


VERDICT -- Etage 0 lexical baseline ICT-35
  F1 lexical baseline        = 0.374 +/- 0.224
  F1 labels shuffled         = 0.503 +/- 0.201
  F1 surface-only (borne)    = 0.286
  Delta (reel - shuffled)    = -0.129

  VERDICT = INCONCLUSIVE borne
  RAISON  = Le delta reel-shuffled < 0.05 BORN la discrimination : sur 30 paires et 6 features lexicales (n_tokens, ponctuation, verbes, callbacks, stop-words, repetition), on ne discrimine pas l'humour reussi de l'unfun au-dela du ratio de classe. Cela ne falsifie PAS l'humour lexical en general -- cela borne la portee de ces 6 features sur ce corpus. Un verdict sur d'autres corpus, ou avec plus de paires, ou avec des features d'ordre superieur (embeddings), reste ouvert.

LIMITES DE CE VERDICT (reformulees -- ce n'est PAS une falsification)
1. 30 paires consommees (1:1 honnete, depuis PAIRES_30). Strict minimum du substrat.
2. Verdict borne a ces 6 features (n_tokens, ponctuation, verbes, callbacks, stop, repetition).
3. Labellisation Argumentu

In [12]:
# Sortie minimale pour les etages superieurs (gated par #5635 GPU Gate 24 + #14033 substrat taxonomique).
#
# IMPORTANT : ce notebook ne SORT PAS de features pour les etages suivants tel quel -- il
# BORNE un signal lexical sur 6 features deterministes, consommees sur 30 paires (1:1 honnete).
# Les etages superieurs (embeddings, modele generatif avec contexte narratif, SAE/J-lens)
# doivent re-apprendre sur des substrats plus expressifs (cf taxonomie sourcee dans le
# frontmatter : capacites 2 a 6 -- jeu sonore seul est couvert par l'etage 0).
if verdict == "FEATURE_CANDIDATE":
    print("[P2] Le verdict autorise une replique avec plus de paires (>100) sur ce meme")
    print("     substrat lexical. Phase 2 SAE peut comparer embeddings + SAE sur les memes")
    print("     30 paires, le gain attendu par rapport au F1 = %.3f est marginal car le" % f1_real)
    print("     lexique est deja capture.")
elif verdict == "SURFACE_SEULE":
    print("[P2] Le verdict BORN : sur ces 6 features, le strip ne casse pas la discrimination.")
    print("     Les etages superieurs (embeddings, contexte, SAE) peuvent detecter ce que le")
    print("     lexique rate, mais ne sont pas garantis d'aboutir. A confirmer sur >= 2 modeles.")
else:
    print("[P2] Le verdict est INCONCLUSIVE borne : les 6 features lexicales ne discriminent")
    print("     pas au-dela du ratio de classe sur 30 paires. Un etage superieur (embeddings,")
    print("     contexte narratif, SAE) peut seul trancher. Ce notebook n'a pas la pretention")
    print("     de trancher au-dela de l'etage 0 lexical.")
print()
print("[P2] Dans tous les cas, les etages superieurs restent gated par #5635 (GPU Gate 24")
print("     workspace) et par #14033 (substrat taxonomique). Aucune action GPU ou")
print("     embeddings n'est prise par ce notebook.")


[P2] Le verdict est INCONCLUSIVE borne : les 6 features lexicales ne discriminent
     pas au-dela du ratio de classe sur 30 paires. Un etage superieur (embeddings,
     contexte narratif, SAE) peut seul trancher. Ce notebook n'a pas la pretention
     de trancher au-dela de l'etage 0 lexical.

[P2] Dans tous les cas, les etages superieurs restent gated par #5635 (GPU Gate 24
     workspace) et par #14033 (substrat taxonomique). Aucune action GPU ou
     embeddings n'est prise par ce notebook.


## Voir aussi

- **#14035** -- issue de reference, protocole falsifiable, acceptance (cible haute).
- **#14032** -- consolidation de la methode de labellisation (GT-28 / GT-28b).
- **#14033** -- substrat taxonomique croissant (corpus apparie + 3 labels + taxonomie sourcee).
- **`MyIA.AI.Notebooks/GameTheory/GameTheory-28b-Humour-Banc-Dur.ipynb`** -- CORPUS_DUR 120 instances, source de verite.
- **`MyIA.AI.Notebooks/IIT/ICT-Series/ICT-21-SAETrajectories.ipynb`** -- substrat S4 (SAE Qwen3.5-9B), aval + tard.
- **`MyIA.AI.Notebooks/IIT/ICT-Series/ICT-24-WorkspaceIgnition.ipynb`** -- Gate 24, gated etages superieurs (GPU).
- **#5635, #8236, #5105, #4588** -- aval ICT-Series.

- **Note :** ICT-35 -- *HumorCausalProbe-Pilot*, sub-grain de #14035, **etage 0 lexical baseline**, GPU-free. Verdict INCONCLUSIVE borne sur 30 paires (consommation honnete de PAIRES_30). L'etape d'apres est #14033 (substrat taxonomique), pas Phase 2 SAE. Si le verdict devient FEATURE_CANDIDATE apres elargissement (>100 paires) ou changement d'etage, ce notebook sera promu en ICT-35 numerote definitif ; sinon il restera un -Pilot archive pour tracabilite.
